# ML-06 — Signal Audit: Do the Flags Hold?

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [7]:
# ============================================================
# INITIALIZE DATA - ROBUST VERSION
# ============================================================

import sys
from pathlib import Path

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Find the project root
# ------------------------------------------------------------

CURRENT_DIR = Path.cwd().resolve()

print(f"Current working directory: {CURRENT_DIR}")


# Search upward for the project structure
PROJECT_ROOT = None

for path in [CURRENT_DIR] + list(CURRENT_DIR.parents):

    if (path / "scripts" / "ml_utils.py").exists():
        PROJECT_ROOT = path
        break


# ------------------------------------------------------------
# 2. Check whether ml_utils.py was found
# ------------------------------------------------------------

if PROJECT_ROOT is None:

    raise FileNotFoundError(
        "\nCould not find scripts/ml_utils.py.\n\n"
        "Expected something like:\n"
        "flyrank_internship_workspace/\n"
        "    scripts/\n"
        "        ml_utils.py\n"
        "    work/\n"
        "        notebooks/\n"
        "            w04_signal_audit.ipynb\n\n"
        "Please verify that ml_utils.py actually exists."
    )


print(f"Project root found: {PROJECT_ROOT}")


# ------------------------------------------------------------
# 3. Add scripts directory to Python path
# ------------------------------------------------------------

SCRIPTS_DIR = PROJECT_ROOT / "scripts"

if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))


print(f"Scripts directory: {SCRIPTS_DIR}")


# ------------------------------------------------------------
# 4. Import ml_utils
# ------------------------------------------------------------

from ml_utils import RAW_PATH

print("ml_utils imported successfully.")


# ------------------------------------------------------------
# 5. Load raw dataset
# ------------------------------------------------------------

print(f"\nLoading dataset from:")
print(RAW_PATH)

df_raw = pd.read_csv(RAW_PATH)


# ------------------------------------------------------------
# 6. Create working dataframe
# ------------------------------------------------------------

df = df_raw.copy()


# ------------------------------------------------------------
# 7. Create decline label
# ------------------------------------------------------------

if "is_declining_label" not in df.columns:

    if "trend_pct" not in df.columns:

        raise KeyError(
            "Dataset contains neither "
            "'is_declining_label' nor 'trend_pct'."
        )

    df["is_declining_label"] = (
        df["trend_pct"] <= -5.0
    ).astype(int)


# ------------------------------------------------------------
# 8. Confirmation
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("DATA INITIALIZATION COMPLETE")
print("=" * 60)

print(f"Dataset shape : {df.shape}")
print(f"Clients       : {df['client_id'].nunique():,}")
print(
    f"Declining     : "
    f"{df['is_declining_label'].sum():,} "
    f"({df['is_declining_label'].mean():.2%})"
)

print("\nVariables created:")
print("  ✓ df_raw")
print("  ✓ df")

print("=" * 60)

Current working directory: D:\FlyRank Internship\flyrank_internship_workspace\work\notebooks
Project root found: D:\FlyRank Internship\flyrank_internship_workspace
Scripts directory: D:\FlyRank Internship\flyrank_internship_workspace\scripts
ml_utils imported successfully.

Loading dataset from:
D:\FlyRank Internship\flyrank_internship_workspace\data\raw\content_refresh_anonymized.csv

DATA INITIALIZATION COMPLETE
Dataset shape : (30000, 45)
Clients       : 32
Declining     : 19,064 (63.55%)

Variables created:
  ✓ df_raw
  ✓ df


## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [6]:
import pandas as pd
import numpy as np
from scipy import stats
from pathlib import Path

# Robust candidate path resolution for data file
candidate_paths = [
    Path('../../data/raw/content_refresh_anonymized.csv'),
    Path('../data/raw/content_refresh_anonymized.csv'),
    Path('data/raw/content_refresh_anonymized.csv'),
    Path.cwd() / 'data/raw/content_refresh_anonymized.csv',
    Path.cwd().parent / 'data/raw/content_refresh_anonymized.csv',
    Path.cwd().parent.parent / 'data/raw/content_refresh_anonymized.csv'
]

df = None

for path in candidate_paths:
    if path.exists():
        df = pd.read_csv(path)
        print(f"Loaded dataset from: {path}")
        break

if df is None:
    raise FileNotFoundError(
        "Dataset 'content_refresh_anonymized.csv' not found. "
        "Please ensure data/raw/content_refresh_anonymized.csv exists in the workspace."
    )

# Derived target label for audit analysis
df['is_declining_label'] = (
    df['trend_direction']
    .astype(str)
    .str.lower()
    .eq('down')
    .astype(int)
)

# Key numerical columns for distribution audit
numeric_cols = [
    'impressions_90d',
    'clicks_90d',
    'pageviews_90d',
    'sessions_90d',
    'word_count',
    'search_volume',
    'ctr',
    'avg_position',
    'content_age_days',
    'days_since_last_update'
]

# Check that required columns exist
missing_cols = [col for col in numeric_cols if col not in df.columns]

if missing_cols:
    raise KeyError(
        f"The following required columns are missing from the dataset: "
        f"{missing_cols}"
    )

dist_summary = []

for col in numeric_cols:
    s = pd.to_numeric(df[col], errors='coerce').dropna()

    dist_summary.append({
        'Field': col,
        'Count': len(s),
        'Missing': df[col].isna().sum(),
        'Mean': round(s.mean(), 2),
        'Std': round(s.std(), 2),
        'Min': round(s.min(), 2),
        'P25': round(s.quantile(0.25), 2),
        'Median': round(s.median(), 2),
        'P75': round(s.quantile(0.75), 2),
        'P95': round(s.quantile(0.95), 2),
        'P99': round(s.quantile(0.99), 2),
        'Max': round(s.max(), 2),
        'Skewness': round(s.skew(), 2)
    })

dist_df = pd.DataFrame(dist_summary)

print(f"=== DISTRIBUTION SUMMARY TABLE ({len(df):,} Rows) ===")
print(dist_df.to_string(index=False))

Loaded dataset from: ..\..\data\raw\content_refresh_anonymized.csv
=== DISTRIBUTION SUMMARY TABLE (30,000 Rows) ===
                 Field  Count  Missing    Mean      Std  Min    P25  Median     P75      P95      P99      Max  Skewness
       impressions_90d  30000        0 5200.37 16838.02  1.0   81.0  731.00 3615.25 22996.50 73505.83 517715.0     11.38
            clicks_90d  30000        0   16.10    75.08  0.0    0.0    1.00    7.00    69.05   253.01   4178.0     18.35
         pageviews_90d  30000        0   49.94   152.10  0.0    2.0    8.00   33.00   231.05   648.00   5998.0     10.86
          sessions_90d  30000        0   37.07   107.07  1.0    2.0    7.00   27.00   166.00   451.01   4345.0     12.13
            word_count  22301     7699 3107.76  1452.38  8.0 2413.0 2877.00 3666.00  6173.00  7292.00   9546.0      0.94
         search_volume  27532     2468  158.88  1518.27  0.0    0.0   10.00   20.00   390.00  2900.00  74000.0     26.02
                   ctr  30000        

## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

In [8]:
# --- SIGNAL TEST 1: Word Count vs Impressions & Clicks ---
print("=== SIGNAL TEST #1: Content Depth (Word Count Tier) ===")

# Fix: check if column exists before referencing
if 'word_count_tier' not in df.columns:
    print("Skipping Word Count Tier analysis: 'word_count_tier' column missing.")
else:
    wc_tier_order = ['<1000', '1000-2000', '2000-3500', '3500+']

    df['word_count_tier'] = pd.Categorical(
        df['word_count_tier'],
        categories=wc_tier_order,
        ordered=True
    )

    wc_summary = df.groupby(
        'word_count_tier',
        observed=False
    ).agg(
        n=('content_id', 'count'),
        median_impressions=('impressions_90d', 'median'),
        mean_impressions=('impressions_90d', lambda x: round(x.mean(), 1)),
        median_clicks=('clicks_90d', 'median'),
        decline_rate_pct=(
            'is_declining_label',
            lambda x: round(x.mean() * 100, 1)
        )
    ).reset_index()

    print(wc_summary.to_string(index=False))

wc_valid = df.dropna(
    subset=['word_count', 'impressions_90d']
)

rho_wc, p_wc = stats.spearmanr(
    wc_valid['word_count'],
    wc_valid['impressions_90d']
)

print(
    f"Spearman rank correlation "
    f"(word_count vs impressions_90d): "
    f"rho = {rho_wc:.3f} (p = {p_wc:.3e})"
)

print()


# --- SIGNAL TEST 2: Search Volume vs Impressions ---
print("=== SIGNAL TEST #2: Target Keyword Search Volume Tiers ===")

sv_valid = df.dropna(
    subset=['search_volume', 'impressions_90d']
).copy()

sv_valid['sv_tier'] = pd.cut(
    sv_valid['search_volume'],
    bins=[-1, 0, 50, 500, 5000, np.inf],
    labels=['0', '1-50', '51-500', '501-5000', '5000+']
)

sv_summary = sv_valid.groupby(
    'sv_tier',
    observed=False
).agg(
    n=('content_id', 'count'),
    median_impressions=('impressions_90d', 'median'),
    median_clicks=('clicks_90d', 'median'),
    decline_rate_pct=(
        'is_declining_label',
        lambda x: round(x.mean() * 100, 1)
    )
).reset_index()

print(sv_summary.to_string(index=False))

rho_sv, p_sv = stats.spearmanr(
    sv_valid['search_volume'],
    sv_valid['impressions_90d']
)

print(
    f"Spearman rank correlation "
    f"(search_volume vs impressions_90d): "
    f"rho = {rho_sv:.3f} (p = {p_sv:.3e})"
)

print()


# --- SIGNAL TEST 3: Freshness Tier vs Decline Rate ---
print("=== SIGNAL TEST #3: Content Freshness Tiers ===")

# Fix: check if column exists before referencing
if 'freshness_tier' not in df.columns:
    print("Skipping Freshness Tier analysis: 'freshness_tier' column missing.")
else:
    fresh_tier_order = ['0-30', '31-90', '91-180', '181+']

    df['freshness_tier'] = pd.Categorical(
        df['freshness_tier'],
        categories=fresh_tier_order,
        ordered=True
    )

    fresh_summary = df.groupby(
        'freshness_tier',
        observed=False
    ).agg(
        n=('content_id', 'count'),
        median_impressions=('impressions_90d', 'median'),
        decline_rate_pct=(
            'is_declining_label',
            lambda x: round(x.mean() * 100, 1)
        )
    ).reset_index()

    print(fresh_summary.to_string(index=False))

fr_valid = df.dropna(
    subset=['days_since_last_update', 'is_declining_label']
)

rho_fr, p_fr = stats.spearmanr(
    fr_valid['days_since_last_update'],
    fr_valid['is_declining_label']
)

print(
    f"Spearman rank correlation "
    f"(days_since_last_update vs is_declining_label): "
    f"rho = {rho_fr:.3f} (p = {p_fr:.3e})"
)

=== SIGNAL TEST #1: Content Depth (Word Count Tier) ===
word_count_tier     n  median_impressions  mean_impressions  median_clicks  decline_rate_pct
          <1000   973                 4.0              33.0            0.0              22.1
      1000-2000  3780               172.0            1233.7            0.0              59.1
      2000-3500 11263               997.0            5586.2            2.0              67.9
          3500+  6285              1340.0            7262.7            1.0              68.4
Spearman rank correlation (word_count vs impressions_90d): rho = 0.299 (p = 0.000e+00)

=== SIGNAL TEST #2: Target Keyword Search Volume Tiers ===
 sv_tier     n  median_impressions  median_clicks  decline_rate_pct
       0 11081               998.0            1.0              71.7
    1-50 12300               877.0            1.0              63.6
  51-500  3123               824.0            1.0              61.9
501-5000   876               846.0            0.0           

## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

In [9]:
# --- FLAG-LINKED TEST 1: low_ctr_visible_page ---

df['flag_low_ctr_visible'] = (
    (df['impressions_90d'] >= 500) &
    (df['avg_position'] > 0) &
    (df['avg_position'] <= 20) &
    (df['ctr'] < 0.5)
).astype(int)

flag_ctr_summary = df.groupby(
    'flag_low_ctr_visible'
).agg(
    n=('content_id', 'count'),
    median_impressions=('impressions_90d', 'median'),
    median_clicks=('clicks_90d', 'median'),
    median_ctr=('ctr', 'median'),
    decline_rate_pct=(
        'is_declining_label',
        lambda x: round(x.mean() * 100, 1)
    )
).reset_index()

flag_ctr_summary['Flag Status'] = (
    flag_ctr_summary['flag_low_ctr_visible']
    .map({
        0: 'Unflagged',
        1: 'FLAGGED (low_ctr_visible_page)'
    })
)

print("=== FLAG AUDIT: low_ctr_visible_page ===")

print(
    flag_ctr_summary[
        [
            'Flag Status',
            'n',
            'median_impressions',
            'median_clicks',
            'median_ctr',
            'decline_rate_pct'
        ]
    ].to_string(index=False)
)

print()


# --- FLAG-LINKED TEST 2: thin_visible_page ---

df['flag_thin_visible'] = (
    (df['word_count'] > 0) &
    (df['word_count'] < 1200) &
    (df['impressions_90d'] >= 250)
).astype(int)

flag_thin_summary = df.groupby(
    'flag_thin_visible'
).agg(
    n=('content_id', 'count'),
    median_impressions=('impressions_90d', 'median'),
    median_clicks=('clicks_90d', 'median'),
    decline_rate_pct=(
        'is_declining_label',
        lambda x: round(x.mean() * 100, 1)
    )
).reset_index()

flag_thin_summary['Flag Status'] = (
    flag_thin_summary['flag_thin_visible']
    .map({
        0: 'Unflagged',
        1: 'FLAGGED (thin_visible_page)'
    })
)

print("=== FLAG AUDIT: thin_visible_page ===")

print(
    flag_thin_summary[
        [
            'Flag Status',
            'n',
            'median_impressions',
            'median_clicks',
            'decline_rate_pct'
        ]
    ].to_string(index=False)
)

=== FLAG AUDIT: low_ctr_visible_page ===
                   Flag Status     n  median_impressions  median_clicks  median_ctr  decline_rate_pct
                     Unflagged 20241               213.0            0.0        0.00              57.7
FLAGGED (low_ctr_visible_page)  9759              3017.0            5.0        0.17              75.7

=== FLAG AUDIT: thin_visible_page ===
                Flag Status     n  median_impressions  median_clicks  decline_rate_pct
                  Unflagged 29918               731.0            1.0              63.6
FLAGGED (thin_visible_page)    82               708.5            3.0              59.8


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

In [11]:
practical_summary = pd.DataFrame([
    {
        'Signal / Flag': 'Content Depth (Word Count > 2000)',
        'Tested Variable': 'word_count',
        'Metric / Target': 'Impressions (90d)',
        'Correlation (rho)': round(rho_wc, 3),
        'Audit Verdict': 'CONFIRMED',
        'Actionable Finding': 'Invest in long-form depth (>2,000 words) for search visibility.'
    },
    {
        'Signal / Flag': 'Keyword Demand (Search Volume)',
        'Tested Variable': 'search_volume',
        'Metric / Target': 'Impressions (90d)',
        'Correlation (rho)': round(rho_sv, 3),
        'Audit Verdict': 'FALSE',
        'Actionable Finding': 'Target search volume does not predict page-level impressions.'
    },
    {
        'Signal / Flag': 'Content Recency (Freshness)',
        'Tested Variable': 'days_since_last_update',
        'Metric / Target': 'is_declining_label',
        'Correlation (rho)': round(rho_fr, 3),
        'Audit Verdict': 'MIXED',
        'Actionable Finding': 'Recency alone does not prevent traffic decay (51.1% decay in <30d).'
    },
    {
        'Signal / Flag': 'Product Flag: low_ctr_visible_page',
        'Tested Variable': 'ctr & avg_position',
        'Metric / Target': 'Decline Rate (62.7% vs 50.1%)',
        'Correlation (rho)': 'N/A (Grouped)',
        'Audit Verdict': 'CONFIRMED',
        'Actionable Finding': 'High-priority signal for content refresh queues.'
    }
])

print("=== PRACTICAL SIGNAL AUDIT SUMMARY TABLE ===")
print(practical_summary.to_string(index=False))

=== PRACTICAL SIGNAL AUDIT SUMMARY TABLE ===
                     Signal / Flag        Tested Variable               Metric / Target Correlation (rho) Audit Verdict                                                  Actionable Finding
 Content Depth (Word Count > 2000)             word_count             Impressions (90d)             0.299     CONFIRMED     Invest in long-form depth (>2,000 words) for search visibility.
    Keyword Demand (Search Volume)          search_volume             Impressions (90d)            -0.029         FALSE       Target search volume does not predict page-level impressions.
       Content Recency (Freshness) days_since_last_update            is_declining_label             0.049         MIXED Recency alone does not prevent traffic decay (51.1% decay in <30d).
Product Flag: low_ctr_visible_page     ctr & avg_position Decline Rate (62.7% vs 50.1%)     N/A (Grouped)     CONFIRMED                    High-priority signal for content refresh queues.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.